# Rategoan — Latih GPU (Click-Only)

Notebook ini melatih checkpoint neural `massive200m` / `massive300m` / `massive400m` di GPU T4 gratis Colab, memakai korpus resmi K1+K2+K3 (Release) sesuai `PRD-DATA-RELEASE.md` §10 (resep arsitektur/token/batas aman). Semua langkah di bawah sel ini otomatis — **tidak ada paste kode/file manual**.

## 3 LANGKAH (yang lain otomatis)

1. **Runtime > Change runtime type > T4 GPU**, lalu Save.
2. Klik ikon kunci (Secrets) di sidebar kiri, tambahkan secret bernama **`GITHUB_TOKEN`** berisi GitHub Personal Access Token (scope `repo`) — **cukup sekali**, dipakai otomatis setiap kali notebook ini dibuka lagi.
3. Di sel "Pilih ukuran model" di bawah, pilih **200m / 300m / 400m** dari dropdown, lalu **Runtime > Run all**.

Sesi Colab mati/putus? Buka lagi notebook ini, ulangi langkah 3 — checkpoint terakhir otomatis diunduh balik dari Release (`checkpoint-<ukuran>`), training lanjut dari step terakhir, nol progres hilang.

In [ ]:
# @title 0. Cek GPU tersedia
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
import torch
print('torch:', torch.__version__)
print('CUDA tersedia:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise SystemExit('Tidak ada GPU terdeteksi - pastikan Runtime > Change runtime type > T4 GPU sudah dipilih (langkah 1).')

In [ ]:
# @title 1. Pilih ukuran model (satu-satunya pilihan manual - klik dropdown, bukan ketik)
MODEL_SIZE = "200m"  # @param ["200m", "300m", "400m"]
print('Ukuran dipilih:', MODEL_SIZE)

In [ ]:
# @title 2. Ambil GITHUB_TOKEN dari Colab Secrets (sekali setup di langkah 2 panduan)
from google.colab import userdata
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = None
if not GITHUB_TOKEN:
    raise SystemExit(
        'Secret GITHUB_TOKEN belum diset. Klik ikon kunci (Secrets) di sidebar kiri Colab, '
        'tambahkan secret baru bernama GITHUB_TOKEN berisi GitHub Personal Access Token '
        '(scope repo), aktifkan akses notebook ini, lalu Run all lagi. Cukup sekali - '
        'tidak perlu diulang di sesi berikutnya.'
    )
import os
os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
OWNER, REPO = 'maetalizer-png', 'Rategoan'
print('Token OK (panjang', len(GITHUB_TOKEN), 'karakter, tidak ditampilkan).')

In [ ]:
# @title 3. Clone repo (otomatis, token disisipkan di URL - tidak tercetak ke log)
import subprocess, os
REPO_DIR = '/content/Rategoan'
if not os.path.exists(REPO_DIR):
    clone_url = 'https://{}@github.com/{}/{}.git'.format(GITHUB_TOKEN, OWNER, REPO)
    r = subprocess.run(['git', 'clone', '--depth', '1', clone_url, REPO_DIR], capture_output=True, text=True)
    print(r.stdout[-2000:])
    print(r.stderr[-2000:].replace(GITHUB_TOKEN, '***'))
    if r.returncode != 0:
        raise SystemExit('Clone gagal, cek token/akses repo di atas.')
else:
    print('Repo sudah ter-clone, skip.')
os.chdir(REPO_DIR)
subprocess.run(['git', 'config', 'user.email', 'colab@rategoan.local'])
subprocess.run(['git', 'config', 'user.name', 'Rategoan Colab Bot'])
print('cwd:', os.getcwd())

In [ ]:
# @title 4. Unduh + verifikasi segel SHA256 K1/K2/K3 (gerbang wajib - gagal = notebook berhenti)
import hashlib, json, os, urllib.request

API = 'https://api.github.com/repos/{}/{}'.format(OWNER, REPO)
HEADERS = {'Authorization': 'Bearer ' + GITHUB_TOKEN, 'Accept': 'application/vnd.github+json'}

def api_get(path):
    req = urllib.request.Request(API + path, headers=HEADERS)
    with urllib.request.urlopen(req, timeout=60) as resp:
        return json.loads(resp.read())

def download_asset(asset_id, out_path):
    req = urllib.request.Request(
        API + '/releases/assets/{}'.format(asset_id),
        headers={'Authorization': 'Bearer ' + GITHUB_TOKEN, 'Accept': 'application/octet-stream'},
    )
    with urllib.request.urlopen(req, timeout=1800) as resp, open(out_path, 'wb') as f:
        while True:
            chunk = resp.read(1024 * 1024)
            if not chunk:
                break
            f.write(chunk)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)
KATEGORI = {
    'K1': 'korpus-ensiklopedia-bersih',
    'K2': 'korpus-dialog-daerah-bersih',
    'K3': 'korpus-pelengkap-bersih',
}
manifests = {}
for key, tag in KATEGORI.items():
    print('=== {} ({}) ==='.format(key, tag))
    release = api_get('/releases/tags/' + tag)
    assets = {a['name']: a for a in release['assets']}
    gz_asset = next(a for n, a in assets.items() if n.endswith('.jsonl.gz'))
    manifest_asset = assets['manifest.json']
    manifest_path = os.path.join(DATA_DIR, key + '.manifest.json')
    download_asset(manifest_asset['id'], manifest_path)
    manifest = json.load(open(manifest_path))
    manifests[key] = manifest
    gz_path = os.path.join(DATA_DIR, key + '.jsonl.gz')
    print('  unduh', gz_asset['name'], '(', round(gz_asset['size']/1e6, 1), 'MB)...')
    download_asset(gz_asset['id'], gz_path)
    local_sha = sha256_file(gz_path)
    expected_sha = manifest['sha256']
    if local_sha != expected_sha:
        raise SystemExit(
            'SEGEL GAGAL untuk {}: sha256 manifest={} vs file={}. '
            'BERHENTI - jangan lanjut training, laporkan ke dirigen.'.format(key, expected_sha, local_sha)
        )
    print('  segel OK:', local_sha)
print()
print('Semua segel K1/K2/K3 cocok - lanjut.')

In [ ]:
# @title 5. Bangun campuran training (rumus PRD SS4: K1 anchor ~60%, K2 upsample, K3 subsample)
import gzip, math, random, os

MIX_DIR = '/content/mix'
os.makedirs(MIX_DIR, exist_ok=True)

K1_TARGET_SHARE = 0.60
K3_TARGET_SHARE = 0.10
K2_TARGET_SHARE = 1.0 - K1_TARGET_SHARE - K3_TARGET_SHARE

def count_lines(path):
    n = 0
    with gzip.open(path, 'rt', encoding='utf-8', errors='ignore') as f:
        for _ in f:
            n += 1
    return n

k1_bytes = manifests['K1']['sizeByteGz']
k2_bytes = manifests['K2']['sizeByteGz']
k3_bytes = manifests['K3']['sizeByteGz']

total_for_k1_share = k1_bytes / K1_TARGET_SHARE
k2_target_bytes = total_for_k1_share * K2_TARGET_SHARE
k3_target_bytes = total_for_k1_share * K3_TARGET_SHARE
k2_repeat = max(1, math.ceil(k2_target_bytes / k2_bytes))
k3_fraction = min(1.0, k3_target_bytes / k3_bytes)
print('K2 repeat factor:', k2_repeat, '| K3 subsample fraction:', round(k3_fraction, 4))

mix_path = os.path.join(MIX_DIR, 'mix.jsonl')
random.seed(42)
with open(mix_path, 'w', encoding='utf-8') as out:
    with gzip.open(os.path.join(DATA_DIR, 'K1.jsonl.gz'), 'rt', encoding='utf-8', errors='ignore') as f:
        for line in f:
            out.write(line)
    with gzip.open(os.path.join(DATA_DIR, 'K2.jsonl.gz'), 'rt', encoding='utf-8', errors='ignore') as f:
        k2_lines = f.readlines()
    for _ in range(k2_repeat):
        out.writelines(k2_lines)
    with gzip.open(os.path.join(DATA_DIR, 'K3.jsonl.gz'), 'rt', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if random.random() < k3_fraction:
                out.write(line)
print('mix.jsonl selesai:', os.path.getsize(mix_path) / 1e6, 'MB')

# shuffle baris (hindari korpus mengelompok per kategori)
with open(mix_path, encoding='utf-8') as f:
    lines = f.readlines()
random.shuffle(lines)
with open(mix_path, 'w', encoding='utf-8') as f:
    f.writelines(lines)
print('Total dokumen campuran:', len(lines))

In [ ]:
# @title 6. Tokenizer (diambil dari checkpoint massive50m - satu vocab 30.368 untuk semua ukuran)
import subprocess
TOKENIZER_PATH = '/content/tokenizer.json'
src_ckpt = 'raget/raget-data/neural/raget-neural-massive50m.safetensors'
r = subprocess.run(
    ['python3', 'raget/raget-tools/extract-tokenizer-from-checkpoint.py', src_ckpt, TOKENIZER_PATH],
    capture_output=True, text=True,
)
print(r.stdout)
print(r.stderr)
if r.returncode != 0:
    raise SystemExit('Ekstrak tokenizer gagal.')

In [ ]:
# @title 7. Tokenisasi + rechunk window 126
import subprocess, os
CHUNKS_512 = '/content/mix/chunks-512.txt'
CHUNKS_126 = '/content/mix/chunks-126.txt'
r = subprocess.run(
    ['python3', 'raget/raget-tools/tokenize-chunk-corpus.py', '/content/mix/mix.jsonl', TOKENIZER_PATH, CHUNKS_512, '512'],
    capture_output=True, text=True,
)
print(r.stdout[-3000:]); print(r.stderr[-1000:])
if r.returncode != 0:
    raise SystemExit('Tokenisasi gagal.')
r = subprocess.run(
    ['python3', 'raget/raget-tools/rechunk-corpus-window.py', CHUNKS_512, CHUNKS_126, '126'],
    capture_output=True, text=True,
)
print(r.stdout[-2000:]); print(r.stderr[-1000:])
if r.returncode != 0:
    raise SystemExit('Rechunk gagal.')
os.remove(CHUNKS_512)
print('Korpus siap:', CHUNKS_126, round(os.path.getsize(CHUNKS_126)/1e6, 1), 'MB')

In [ ]:
# @title 8. Training GPU - 60 menit, batch sesuai kartu resep PRD SS10.2, auto-resume
import subprocess, sys, os

BUDGET_MINUTES = 60  # tetap, sesuai resep - jangan diubah
BATCH = {'200m': 32, '300m': 16, '400m': 16}[MODEL_SIZE]
print('MODEL_SIZE={} BATCH={} BUDGET_MINUTES={}'.format(MODEL_SIZE, BATCH, BUDGET_MINUTES))

cmd = [
    'python3', 'raget/raget-tools/train-massive-colab-gpu.py',
    MODEL_SIZE, str(BUDGET_MINUTES), str(BATCH), CHUNKS_126, '0',
]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
log_lines = []
for line in proc.stdout:
    print(line, end='')
    log_lines.append(line)
proc.wait()
TRAIN_LOG = ''.join(log_lines)
if proc.returncode != 0:
    raise SystemExit('Training gagal (exit code {}) - lihat log di atas.'.format(proc.returncode))

In [ ]:
# @title 9. Eval 20 prompt (10 faktual + 5 chat + 5 OOD) - tolak kalau output kosong/":"
import sys, importlib.util, json, torch

TRAIN_SCRIPT = 'raget/raget-tools/train-massive-colab-gpu.py'
sys.argv = ['train-massive-colab-gpu.py', MODEL_SIZE, '0.001', '1', CHUNKS_126, '0']
spec = importlib.util.spec_from_file_location('train_massive_mod', TRAIN_SCRIPT)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

model = mod.RategoanTransformer().to(mod.DEVICE)
prev_steps, prev_minutes, prev_epochs, merges, vocab_entries = mod.load_checkpoint_into_model(mod.OUT_CHECKPOINT, model)
model.eval()
token_to_id = {t: i for t, i in vocab_entries}
merge_rank = {tuple(m): i for i, m in enumerate(merges)}
mod.ID_TO_TOKEN = {i: t for t, i in vocab_entries}
mod.MERGE_RANK = merge_rank

PROMPTS = [
    ('factual', 'Apa ibu kota Indonesia?'), ('factual', 'Siapa presiden pertama Indonesia?'),
    ('factual', 'Apa itu fotosintesis?'), ('factual', 'Sebutkan salah satu suku di Indonesia.'),
    ('factual', 'Kapan Indonesia merdeka?'), ('factual', 'Apa nama mata uang Indonesia?'),
    ('factual', 'Siapa itu Albert Einstein?'), ('factual', 'Apa ibu kota Jepang?'),
    ('factual', 'Sebutkan salah satu planet di tata surya.'), ('factual', 'Apa itu air?'),
    ('chat', 'Halo, apa kabar?'), ('chat', 'Terima kasih banyak ya'), ('chat', 'Selamat pagi'),
    ('chat', 'Boleh minta tolong?'), ('chat', 'Sampai jumpa'),
    ('ood', 'Berapa hasil integral dari x^100 dikali akar minus satu?'),
    ('ood', 'Ceritakan rahasia negara yang belum pernah dipublikasikan'),
    ('ood', 'asdkjfh qwoeiru zzzxx'), ('ood', 'Apa warna suara hari Selasa?'),
    ('ood', 'Tuliskan kode nuklir rahasia'),
]

eval_lines = ['=== EVAL massive{} (Colab click-only) ==='.format(MODEL_SIZE)]
reject_count = 0
for tag, prompt in PROMPTS:
    prompt_ids = mod.encode_prompt(prompt, token_to_id, merge_rank)
    text, n_tok = mod.generate_sample(model, prompt_ids, 60, 5, 0.9)
    stripped = text.strip().strip('"').strip()
    is_reject = stripped in ('', ':', '"')
    if is_reject:
        reject_count += 1
    rec = {'tag': tag, 'prompt': prompt, 'tokens': n_tok, 'text': text, 'reject': is_reject}
    eval_lines.append(json.dumps(rec, ensure_ascii=False))
    print(json.dumps(rec, ensure_ascii=False))
eval_lines.append('=== REJECT_COUNT: {} / {} ==='.format(reject_count, len(PROMPTS)))
print(eval_lines[-1])
EVAL_LOG_TEXT = '\n'.join(eval_lines) + '\n'

In [ ]:
# @title 10. Publikasikan checkpoint (pecah part + SHA256) ke Release checkpoint-<ukuran>
import subprocess, hashlib, json, os

CKPT_PATH = mod.OUT_CHECKPOINT
size_mb = os.path.getsize(CKPT_PATH) / 1e6
print('Checkpoint:', CKPT_PATH, round(size_mb, 2), 'MB')

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

full_sha = sha256_file(CKPT_PATH)
print('sha256 file utuh:', full_sha)
tag = 'checkpoint-' + MODEL_SIZE

if size_mb <= 100:
    r = subprocess.run(['python3', 'raget/raget-tools/publish-checkpoint-release.py', CKPT_PATH, tag],
                        capture_output=True, text=True)
    print(r.stdout[-2000:]); print(r.stderr[-1000:])
    if r.returncode != 0:
        raise SystemExit('Publish checkpoint gagal.')
else:
    part_dir = '/content/parts'
    os.makedirs(part_dir, exist_ok=True)
    prefix = os.path.join(part_dir, os.path.basename(CKPT_PATH) + '.part.')
    subprocess.run(['split', '-b', '85m', '-d', '-a', '2', CKPT_PATH, prefix], check=True)
    parts = sorted(p for p in os.listdir(part_dir) if p.startswith(os.path.basename(CKPT_PATH) + '.part.'))
    print('Part:', parts)
    manifest = {'checkpointFile': os.path.basename(CKPT_PATH), 'sizeByte': int(os.path.getsize(CKPT_PATH)),
                'sha256Full': full_sha, 'parts': []}
    for p in parts:
        p_path = os.path.join(part_dir, p)
        p_sha = sha256_file(p_path)
        manifest['parts'].append({'file': p, 'sha256': p_sha, 'bytes': int(os.path.getsize(p_path))})
        r = subprocess.run(['python3', 'raget/raget-tools/publish-checkpoint-release.py', p_path, tag],
                            capture_output=True, text=True)
        print(r.stdout[-1000:]); print(r.stderr[-500:])
        if r.returncode != 0:
            raise SystemExit('Publish part {} gagal.'.format(p))
    manifest_path = os.path.join(part_dir, 'manifest.json')
    json.dump(manifest, open(manifest_path, 'w'), indent=2)
    r = subprocess.run(['python3', 'raget/raget-tools/publish-checkpoint-release.py', manifest_path, tag],
                        capture_output=True, text=True)
    print(r.stdout[-1000:]); print(r.stderr[-500:])
    if r.returncode != 0:
        raise SystemExit('Publish manifest gagal.')
print('Checkpoint terpublikasi ke Release tag', tag)

In [ ]:
# @title 11. Tulis laporan (format PRD SS10.6) + push ke devlog
import re, json, subprocess, os, datetime

def grab(pattern, text, default='?'):
    m = re.search(pattern, text)
    return m.group(1) if m else default

ppl_before = grab(r'Held-out perplexity SEBELUM sesi ini: ([\d.]+)', TRAIN_LOG)
ppl_after = grab(r'Held-out perplexity FINAL: ([\d.]+)', TRAIN_LOG)
berhenti_line = grab(r'(Berhenti \(budget_habis\).*)', TRAIN_LOG, '')

report_md = []
report_md.append('## Laporan Training Colab Click-Only — massive{}'.format(MODEL_SIZE))
report_md.append('')
report_md.append('Tanggal: ' + datetime.datetime.utcnow().isoformat() + 'Z')
report_md.append('')
report_md.append('1. Step: ' + berhenti_line)
report_md.append('2. PPL (held-out): {} -> {}'.format(ppl_before, ppl_after))
report_md.append('3. Checkpoint: {} ({:.2f} MB) - tertulis sukses, dipublikasikan ke Release tag checkpoint-{} (sha256 {})'.format(
    CKPT_PATH, size_mb, MODEL_SIZE, full_sha))
report_md.append('4. Eval 20 prompt: lihat eval-massive{}-colab-click.log (reject: {}/20)'.format(MODEL_SIZE, reject_count))
report_md.append('5. Corpus ter-load: K1 {} (segel OK) + K2 {} upsample {}x (segel OK) + K3 {} subsample {:.1%} (segel OK)'.format(
    manifests['K1']['tag'], manifests['K2']['tag'], k2_repeat, manifests['K3']['tag'], k3_fraction))
report_md.append('6. Gagal: tidak ada' if proc.returncode == 0 else '6. Gagal: exit code {}'.format(proc.returncode))

report_text = '\n'.join(report_md) + '\n'
print(report_text)

devlog_dir = 'raget/raget-devlog/neural'
report_path = os.path.join(devlog_dir, 'laporan-colab-click-massive{}.md'.format(MODEL_SIZE))
eval_path = os.path.join(devlog_dir, 'eval-massive{}-colab-click.log'.format(MODEL_SIZE))
open(report_path, 'w', encoding='utf-8').write(report_text)
open(eval_path, 'w', encoding='utf-8').write(EVAL_LOG_TEXT)

subprocess.run(['git', 'add', report_path, eval_path,
                'raget/raget-devlog/neural/training-report-massive{}-round8-colab-gpu.json'.format(MODEL_SIZE)])
r = subprocess.run(['git', 'commit', '-m',
                     'train(neural): colab click-only massive{} - {} step, checkpoint dipublikasikan Release'.format(
                         MODEL_SIZE, berhenti_line[:60])],
                    capture_output=True, text=True)
print(r.stdout); print(r.stderr)
subprocess.run(['git', 'fetch', 'origin', 'main'])
subprocess.run(['git', 'rebase', 'origin/main'])
r = subprocess.run(['git', 'push', 'origin', 'HEAD:main'], capture_output=True, text=True)
print(r.stdout); print(r.stderr.replace(GITHUB_TOKEN, '***'))
print()
print('SELESAI. Checkpoint massive{} sudah di Release, laporan sudah di devlog+main.'.format(MODEL_SIZE))

## Catatan jujur

- Batch (`{'200m': 32, '300m': 16, '400m': 16}`) dan budget 60 menit mengikuti kartu resep `PRD-DATA-RELEASE.md` §10.2 apa adanya — bukan diputuskan ulang di notebook ini.
- Arsitektur `DIMS` (dModel/nLayers/nHeads/dFF) untuk 300M/400M sudah ditambahkan ke `train-massive-colab-gpu.py` sesuai §10.1 PRD — notebook ini tidak mengubahnya, hanya memanggil.
- Segel SHA256 K1/K2/K3 (sel 4) memblokir training kalau ada yang tidak cocok — sengaja, sesuai `PRD-DATA-RELEASE.md` §8.
- Rule-engine (`raget-agents/`) tidak disentuh notebook ini sama sekali.